# ICU Transfer Prediction — Support Vector Machine (SVM)

**Dataset:** cleaned MIMIC-IV-derived admission features provided with this project.  
**Target:** `icu_transfer` (1 = ICU transfer, 0 = no ICU transfer).

This notebook uses the **existing train/test split** and never uses the test set for model selection or threshold selection. It fits preprocessing inside scikit-learn pipelines to avoid validation leakage.

**Important clinical caveat:** These are retrospective labels and engineered predictors. The notebook does not establish whether each predictor was available *before* the intended prediction time. Check temporal leakage, patient overlap between splits, and the assumptions used to construct the kidney-disease cohort before interpreting performance clinically.

In [ ]:
import importlib.util
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

libraries = [
    "pathlib",
    "json",
    "warnings",
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "joblib",
    "sklearn"
]

for lib in libraries:
    if importlib.util.find_spec(lib) is not None:
        print(f"✓ {lib}: Installed")
    else:
        print(f"✗ {lib}: NOT INSTALLED")

In [ ]:
%pip install numpy pandas matplotlib seaborn joblib scikit-learn

## 1. Imports and reproducibility
Install dependencies with `pip install numpy pandas scipy scikit-learn matplotlib seaborn joblib` if necessary.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report,
    precision_score, recall_score, f1_score, balanced_accuracy_score,
    PrecisionRecallDisplay, RocCurveDisplay)

RANDOM_STATE = 42
TARGET = 'icu_transfer'
OUTPUT_DIR = Path('svm_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Ready. scikit-learn pipelines keep scaling inside training folds.')

## 2. Load and validate the supplied datasets
Place both CSV files in the same directory as this notebook, or change `DATA_DIR`.

In [ ]:
DATA_DIR = Path('.')
train_path = DATA_DIR / 'kidney_final_train_dataset.csv'
test_path = DATA_DIR / 'kidney_final_test_dataset.csv'
assert train_path.exists() and test_path.exists(), 'CSV files not found. Update DATA_DIR.'
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
assert TARGET in train_df.columns and TARGET in test_df.columns
assert list(train_df.columns) == list(test_df.columns), 'Train/test columns differ or are reordered.'
assert not train_df.columns.duplicated().any(), 'Duplicate column names.'
assert set(train_df[TARGET].dropna().unique()) == {0,1}
assert set(test_df[TARGET].dropna().unique()) == {0,1}
feature_cols = [c for c in train_df.columns if c != TARGET]
X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET].astype(int)
X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET].astype(int)
assert len(feature_cols) == 20
assert all(pd.api.types.is_numeric_dtype(t) for t in X_train.dtypes)
assert np.isfinite(X_train.to_numpy()).all() and np.isfinite(X_test.to_numpy()).all()
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Train label.', y_train.value_counts().sort_index())
print('Test label.', y_test.value_counts().sort_index())
print('Train prevalence:', f'{y_train.mean():.3%}')
print('Test prevalence:', f'{y_test.mean():.3%}')
display(pd.DataFrame({'feature': feature_cols, 'dtype': X_train.dtypes.astype(str).values}))

## 3. Evaluation design

- **LinearSVC:** efficient linear decision boundary and interpretable coefficient direction.
- **RBF SVC:** nonlinear decision boundary; can be much slower at this dataset size.
- **StandardScaler:** essential because SVM margins/distances are scale-dependent. It is fit *only on training data* within each fold.
- **`class_weight="balanced"`:** penalizes minority-class mistakes more heavily; this is a starting point, not a guarantee of better precision or calibration.
- **Average precision (AP):** primary tuning score, reflecting precision-recall performance for rare ICU transfers.
- **Threshold:** selected from a training-only validation split. The test set is evaluated once after selection.

We use a **stratified 80/20 split within the supplied training set** for model development and threshold selection. After selecting hyperparameters and threshold, we refit on the entire supplied training set. This keeps the supplied test set untouched.

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
fit_idx, val_idx = next(splitter.split(X_train, y_train))
X_fit, X_val = X_train.iloc[fit_idx], X_train.iloc[val_idx]
y_fit, y_val = y_train.iloc[fit_idx], y_train.iloc[val_idx]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
print('Fit:', X_fit.shape, 'Validation:', X_val.shape)
print('AP random-ranking baseline ≈ positive prevalence:', round(y_fit.mean(), 4))

## 4. Baseline: class-weighted linear SVM

In [ ]:
linear_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('svm', LinearSVC(C=1.0, class_weight='balanced', max_iter=20000, random_state=RANDOM_STATE))
])
linear_pipe.fit(X_fit, y_fit)
linear_val_scores = linear_pipe.decision_function(X_val)
print('Linear validation AP:', round(average_precision_score(y_val, linear_val_scores), 4))
print('Linear validation ROC-AUC:', round(roc_auc_score(y_val, linear_val_scores), 4))

## 5. RBF SVM hyperparameter search

An RBF SVM can be computationally expensive on ~73k admissions. By default, this search uses a **stratified subset of 12,000 development samples**, then evaluates the best configuration on the full validation split. Increase `RBF_TUNING_SAMPLES` or `N_ITER` only if runtime permits. The chosen model is later refit on all 73,413 training samples.

`C` controls regularization; `gamma` controls how local the nonlinear decision boundary becomes. Both should be tuned, rather than chosen from test performance.

In [ ]:
RBF_TUNING_SAMPLES = 12000  # increase to len(X_fit) for full development-data tuning
N_ITER = 8
if len(X_fit) > RBF_TUNING_SAMPLES:
    sub_split = StratifiedShuffleSplit(n_splits=1, train_size=RBF_TUNING_SAMPLES, random_state=RANDOM_STATE)
    sub_idx, _ = next(sub_split.split(X_fit, y_fit))
    X_tune, y_tune = X_fit.iloc[sub_idx], y_fit.iloc[sub_idx]
else:
    X_tune, y_tune = X_fit, y_fit

rbf_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('svm', SVC(kernel='rbf', class_weight='balanced', cache_size=1000))
])
search = RandomizedSearchCV(
    estimator=rbf_pipe,
    param_distributions={'svm__C': [0.1, 1, 10, 100],
                         'svm__gamma': ['scale', 0.001, 0.01, 0.1]},
    n_iter=N_ITER,
    scoring='average_precision', cv=cv, n_jobs=-1,
    random_state=RANDOM_STATE, refit=True, verbose=1,
    return_train_score=False
)
search.fit(X_tune, y_tune)
print('Best RBF params:', search.best_params_)
print('Mean cross-validation AP:', round(search.best_score_, 4))
display(pd.DataFrame(search.cv_results_)[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
        .sort_values('rank_test_score').head(10))

## 6. Compare models on the held-out validation split

In [ ]:
# Refit the selected RBF configuration on ALL development data (not just the tuning subset).
from sklearn.base import clone
rbf_selected = clone(search.best_estimator_)
rbf_selected.fit(X_fit, y_fit)
rbf_val_scores = rbf_selected.decision_function(X_val)
validation_results = pd.DataFrame([
    {'model': 'Linear SVM', 'average_precision': average_precision_score(y_val, linear_val_scores),
     'roc_auc': roc_auc_score(y_val, linear_val_scores)},
    {'model': 'RBF SVM', 'average_precision': average_precision_score(y_val, rbf_val_scores),
     'roc_auc': roc_auc_score(y_val, rbf_val_scores)}
]).sort_values('average_precision', ascending=False)
display(validation_results)
model_lookup = {'Linear SVM': linear_pipe, 'RBF SVM': rbf_selected}
score_lookup = {'Linear SVM': linear_val_scores, 'RBF SVM': rbf_val_scores}
selected_name = validation_results.iloc[0]['model']
selected_model = model_lookup[selected_name]
selected_scores = score_lookup[selected_name]
print('Selected:', selected_name)
fig, ax = plt.subplots(figsize=(7,5))
for name, scores in score_lookup.items():
    PrecisionRecallDisplay.from_predictions(y_val, scores, name=name, ax=ax)
ax.axhline(y_val.mean(), linestyle='--', color='gray', label='Prevalence baseline')
ax.legend(); ax.set_title('Validation precision-recall curves'); plt.show()

## 7. Choose a clinically meaningful decision threshold

`decision_function` returns **margin scores, not probabilities**. The default SVM threshold is zero. We compare it with a threshold maximizing F1 on validation data, and optionally a recall-constrained threshold.

**Important:** maximizing F1 is only a demonstration. For clinical screening, false-negative and false-positive costs must guide the operating point. Threshold selection uses validation labels only.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, selected_scores)
f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
best_i = int(np.argmax(f1_values))
threshold_f1 = float(thresholds[best_i])
MIN_RECALL = 0.80
eligible = np.where(recall[:-1] >= MIN_RECALL)[0]
threshold_recall = float(thresholds[eligible[np.argmax(precision[eligible])]]) if len(eligible) else None
print('Default SVM threshold: 0.0')
print('Validation F1-optimal threshold:', round(threshold_f1, 5))
print('Threshold for >=80% validation recall:', threshold_recall)

def metrics_at_threshold(y_true, scores, threshold):
    pred = (np.asarray(scores) >= threshold).astype(int)
    return {'threshold': threshold,
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred, zero_division=0),
            'f1': f1_score(y_true, pred, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(y_true, pred),
            'predicted_positive': int(pred.sum())}
threshold_rows = [dict(strategy='Default 0', **metrics_at_threshold(y_val, selected_scores, 0.0)),
                  dict(strategy='Max validation F1', **metrics_at_threshold(y_val, selected_scores, threshold_f1))]
if threshold_recall is not None:
    threshold_rows.append(dict(strategy='Recall >= 80%', **metrics_at_threshold(y_val, selected_scores, threshold_recall)))
display(pd.DataFrame(threshold_rows))
# Change this if your use case prioritizes recall over F1.
FINAL_THRESHOLD = threshold_f1

## 8. Refit the selected model on the complete supplied training dataset

Hyperparameters and threshold are now frozen. This is the first time the selected model is fit on all supplied training admissions. **Do not adjust the model or threshold after viewing test results.**

In [ ]:
final_model = clone(selected_model)
final_model.fit(X_train, y_train)
test_scores = final_model.decision_function(X_test)
test_predictions = (test_scores >= FINAL_THRESHOLD).astype(int)
print('Final model:', selected_name)
print('Frozen threshold:', FINAL_THRESHOLD)

## 9. Final test-set evaluation

In [ ]:
test_metrics = {
    'model': selected_name,
    'average_precision': average_precision_score(y_test, test_scores),
    'roc_auc': roc_auc_score(y_test, test_scores),
    'precision': precision_score(y_test, test_predictions, zero_division=0),
    'recall': recall_score(y_test, test_predictions, zero_division=0),
    'f1': f1_score(y_test, test_predictions, zero_division=0),
    'balanced_accuracy': balanced_accuracy_score(y_test, test_predictions)
}
display(pd.DataFrame([test_metrics]).round(4))
print(classification_report(y_test, test_predictions, target_names=['No ICU transfer', 'ICU transfer'], zero_division=0))
cm = confusion_matrix(y_test, test_predictions, labels=[0,1])
print('Confusion matrix [[TN, FP], [FN, TP]]:\n', cm)
ConfusionMatrixDisplay(cm, display_labels=['No transfer', 'ICU transfer']).plot(cmap='Blues', values_format='d')
plt.title('Held-out test confusion matrix'); plt.show()
fig, axes = plt.subplots(1,2,figsize=(12,5))
PrecisionRecallDisplay.from_predictions(y_test, test_scores, ax=axes[0])
axes[0].axhline(y_test.mean(), color='gray', linestyle='--', label='Prevalence baseline')
axes[0].legend(); axes[0].set_title('Test precision-recall')
RocCurveDisplay.from_predictions(y_test, test_scores, ax=axes[1])
axes[1].plot([0,1],[0,1],linestyle='--',color='gray'); axes[1].set_title('Test ROC')
plt.tight_layout(); plt.show()

## 10. Inspect linear feature coefficients (only if linear model wins)
Coefficients are in standardized feature units. Correlation, collinearity and observational bias prevent causal interpretation.

In [ ]:
if selected_name == 'Linear SVM':
    coefs = pd.Series(final_model.named_steps['svm'].coef_[0], index=feature_cols)
    display(coefs.sort_values(key=abs, ascending=False).to_frame('standardized_coefficient'))
    coefs.sort_values().plot.barh(figsize=(9,7)); plt.title('Linear SVM standardized coefficients'); plt.tight_layout(); plt.show()
else:
    print('RBF SVM does not provide a directly interpretable per-feature coefficient vector.')

## 11. Save the trained pipeline, threshold, metrics, and predictions

The serialized pipeline includes the fitted scaler. `decision_score` is **not** a calibrated probability. Only load `joblib` files from trusted sources.

In [ ]:
joblib.dump({'pipeline': final_model, 'threshold': FINAL_THRESHOLD,
             'feature_columns': feature_cols, 'model_name': selected_name},
            OUTPUT_DIR / 'icu_transfer_svm.joblib')
pd.DataFrame({'true_label': y_test.to_numpy(), 'decision_score': test_scores,
              'predicted_label': test_predictions}).to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
with open(OUTPUT_DIR / 'test_metrics.json', 'w') as f:
    json.dump({k: (float(v) if isinstance(v, (float, np.floating)) else v)
               for k,v in test_metrics.items()}, f, indent=2)
print('Saved artifacts in', OUTPUT_DIR.resolve())

## 12. Interpretation and research validity checklist

1. **Time of prediction:** Are the `latest_value`, `count`, and `change` variables computed exclusively from measurements available *before* the ICU transfer or prediction cutoff? If not, the task suffers from target leakage.
2. **Patient separation:** Were multiple admissions from the same patient assigned to different train/test partitions? Without patient identifiers in these CSVs, this cannot be verified. Prefer patient-grouped splitting upstream.
3. **Kidney cohort:** Verify the source notebook's assumptions about renal disease identification and whether they match the intended study population.
4. **Clinical utility:** Evaluate recall, false alerts per 100 admissions, precision, calibration (if probabilities are later added), and subgroup performance; accuracy alone is misleading.
5. **Model comparison:** Compare against a simple logistic regression baseline on the identical patient-level split. An SVM is not automatically superior.
6. **Statistical uncertainty:** For a publication-grade analysis, report patient-level bootstrap confidence intervals and consider external or temporal validation.
7. **Runtime:** Full RBF fitting can be slow. The tuning subset reduces search cost, not final fitting cost.

**Do not claim prospective clinical validity from a retrospective test score.**